In [ ]:
import json
import os
import numpy as np
from decimal import Decimal, getcontext

# Set high precision for decimal operations
getcontext().prec = 1000


def safe_multiply_encrypted_values(*values):
    if all(isinstance(v, dict) for v in values):
        result = {}
        for k in values[0].keys():
            if all(k in v for v in values):
                result[k] = safe_multiply_encrypted_values(*(v[k] for v in values))
        return result
    elif all(isinstance(v, list) for v in values):
        result = []
        min_len = min(len(v) for v in values)
        for i in range(min_len):
            result.append(safe_multiply_encrypted_values(*(v[i] for v in values)))
        return result
    elif all(isinstance(v, dict) and 'ciphertext' in v for v in values):
        try:
            c_vals = [Decimal(str(v['ciphertext'])) for v in values]
            if not all(c.is_finite() for c in c_vals):
                print("Warning: Non-finite value detected. Skipping multiplication.")
                return values[0]
            result_cipher = Decimal(1)
            for c in c_vals:
                result_cipher *= c
            if result_cipher > Decimal(10 ** 1000):
                print("Warning: Result too large")
                return values[0]
            result = {
                'ciphertext': str(result_cipher),
                'aggregated': True,
                'num_models': len(values)
            }
            return result
        except Exception as e:
            print(f"Error in multiplication: {e}")
            return values[0]
    elif all(isinstance(v, (int, float)) for v in values):
        try:
            result = np.prod(values)
            if not np.isfinite(result):
                return values[0]
            return result
        except:
            return values[0]
    else:
        try:
            f_vals = [float(v) for v in values]
            result = np.prod(f_vals)
            if not np.isfinite(result):
                return values[0]
            return result
        except:
            return values[0]



def homomorphic_add_encrypted_values(*values):
    if all(isinstance(v, dict) for v in values):
        result = {}
        for k in values[0].keys():
            if all(k in v for v in values):
                result[k] = homomorphic_add_encrypted_values(*(v[k] for v in values))
        return result
    elif all(isinstance(v, list) for v in values):
        result = []
        min_len = min(len(v) for v in values)
        for i in range(min_len):
            result.append(homomorphic_add_encrypted_values(*(v[i] for v in values)))
        return result
    elif all(isinstance(v, dict) and 'ciphertext' in v for v in values):
        try:
            c_vals = [Decimal(str(v['ciphertext'])) for v in values]
            if not all(c.is_finite() for c in c_vals):
                return values[0]
            result_cipher = Decimal(1)
            for c in c_vals:
                result_cipher *= c
            if result_cipher > Decimal(10 ** 500):
                print("Warning: Ciphertext too large")
                return values[0]
            return {
                'ciphertext': str(result_cipher),
                'aggregated': True,
                'num_models': len(values),
                'operation': 'homomorphic_addition'
            }
        except:
            return values[0]
    elif all(isinstance(v, (int, float)) for v in values):
        return sum(values)
    else:
        return values[0]


def aggregate_weights(file_list, output_file, method="add"):
    try:
        print("Loading weights from input files...")
        all_weights = []
        for f in file_list:
            with open(f, 'r') as file:
                all_weights.append(json.load(file))

        print(f"Aggregating weights using method: {method}")
        aggregated = {}

        for key in all_weights[0].keys():
            if key.startswith('_'):
                aggregated[key] = all_weights[0][key]
                continue
            if not all(key in w for w in all_weights):
                print(f"Warning: Key '{key}' not in all models. Skipping.")
                continue
            print(f"Aggregating key: {key}")
            values = [w[key] for w in all_weights]
            if method == "add":
                aggregated[key] = homomorphic_add_encrypted_values(*values)
            else:
                aggregated[key] = safe_multiply_encrypted_values(*values)

        aggregated['_aggregation_metadata'] = {
            'method': method,
            'num_models': len(file_list),
            'input_files': file_list,
            'output_file': output_file
        }

        with open(output_file, 'w') as f:
            json.dump(aggregated, f, indent=2)

        print("Aggregation complete.")
        return aggregated
    except Exception as e:
        print(f"Error: {e}")
        return None



def validate_encrypted_file(filename: str):
    """
    Validate that an encrypted weights file doesn't contain infinity values
    """
    print(f"Validating {filename}...")

    try:
        with open(filename, 'r') as f:
            data = json.load(f)

        def check_for_infinity(obj, path=""):
            issues = []
            if isinstance(obj, dict):
                for key, value in obj.items():
                    current_path = f"{path}.{key}" if path else key
                    if key == 'ciphertext':
                        try:
                            val = float(value)
                            if not np.isfinite(val):
                                issues.append(f"Non-finite ciphertext at {current_path}: {value}")
                        except:
                            pass
                    else:
                        issues.extend(check_for_infinity(value, current_path))
            elif isinstance(obj, list):
                for i, item in enumerate(obj):
                    issues.extend(check_for_infinity(item, f"{path}[{i}]"))
            elif isinstance(obj, (int, float)):
                if not np.isfinite(obj):
                    issues.append(f"Non-finite value at {path}: {obj}")
            return issues

        issues = check_for_infinity(data)

        if issues:
            print(f"Found {len(issues)} issues in {filename}:")
            for issue in issues[:10]:  # Show first 10 issues
                print(f"  - {issue}")
            if len(issues) > 10:
                print(f"  ... and {len(issues) - 10} more issues")
        else:
            print(f"✓ {filename} validation passed - no infinity values found")

        return len(issues) == 0

    except Exception as e:
        print(f"Error validating {filename}: {e}")
        return False

current_dir = os.getcwd()
# encrypted_weights_path = rf"{current_dir}\encrypted_weights_2.json"

# Usage
if __name__ == "__main__":
    print("Secure Paillier Weight Aggregation Tool")
    file_list = []  # Initialize empty list
    for i in range(20):
        path = rf"{current_dir}\weights\encrypted_weights_{i+1}.json"
        if os.path.exists(path):
            file_list.append(path)
        else:
            print(f"⚠ File not found and will be skipped: {path}")
    
    if not file_list:
        print("❌ No encrypted weight files found!")
        exit(1)
    
    print(f"✅ Found {len(file_list)} encrypted weight files to aggregate")
    
    output_file = "aggregated_weights_safe.json"

    print("\nValidating input files...")
    all_valid = all(validate_encrypted_file(f) for f in file_list)

    if not all_valid:
        print("\n⚠ Some files contain non-finite values. Proceeding anyway...")

    
    method = "add" 
    
    print(f"\nAggregating using {method}...")
    result = aggregate_weights(file_list, output_file, method)

    if result:
        print("\nValidating output file...")
        validate_encrypted_file(output_file)
        print("\n✅ Aggregation completed!")
        print(f"Output saved to: {output_file}")
    else:
        print("\n❌ Aggregation failed.")

Secure Paillier Weight Aggregation Tool
⚠ File not found and will be skipped: C:\Users\ASUS\the system folder\GS\weights\encrypted_weights_4.json
⚠ File not found and will be skipped: C:\Users\ASUS\the system folder\GS\weights\encrypted_weights_5.json
⚠ File not found and will be skipped: C:\Users\ASUS\the system folder\GS\weights\encrypted_weights_6.json
⚠ File not found and will be skipped: C:\Users\ASUS\the system folder\GS\weights\encrypted_weights_7.json
⚠ File not found and will be skipped: C:\Users\ASUS\the system folder\GS\weights\encrypted_weights_8.json
⚠ File not found and will be skipped: C:\Users\ASUS\the system folder\GS\weights\encrypted_weights_9.json
⚠ File not found and will be skipped: C:\Users\ASUS\the system folder\GS\weights\encrypted_weights_10.json
⚠ File not found and will be skipped: C:\Users\ASUS\the system folder\GS\weights\encrypted_weights_11.json
⚠ File not found and will be skipped: C:\Users\ASUS\the system folder\GS\weights\encrypted_weights_12.json
⚠ F